# Análisis de los resultados

In [1]:
from pyspark.sql import SparkSession, functions as F, types as T
import pandas as pd
import matplotlib.pyplot as plt


spark = (
    SparkSession.builder
    .appName("eda_musicbrainz")
    .config("spark.driver.memory", "2g")
    .enableHiveSupport()
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-25T01:50:27,900 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
ANL = "/Obligatorio/analytics"   # o como hayas nombrado la zona de analíticas
sc = spark.sparkContext
hadoop = sc._jvm.org.apache.hadoop.fs
fs = hadoop.FileSystem.get(sc._jsc.hadoopConfiguration())
fs.mkdirs(hadoop.Path(ANL))

True

## Pregunta 1: festivales con mayor accesibilidad aérea (score del aeropuerto más cercano)

In [3]:
spark.sql("USE festivales_aereo")


q1 = spark.sql("""
    SELECT  f.festival_name,
            c.country_name                AS pais,
            ROUND(a.distance_km, 1)       AS dist_aeropuerto_km,
            a.direct_countries            AS paises_directos,
            a.airlines_count              AS aerolineas,
            ROUND(a.connectivity_score,1) AS conectividad,
            ROUND(a.accessibility_score,1) AS accesibilidad
    FROM fact_festival_air_accessibility a
    JOIN dim_festival f  ON a.festival_id = f.festival_id
    LEFT JOIN dim_country c ON f.country_id = c.country_id
    WHERE a.airport_rank = 1                 -- aeropuerto más cercano a cada festival
    ORDER BY a.accessibility_score DESC
    LIMIT 20
""")
q1.show(20, truncate=False)


2026-06-25T01:50:43,160 INFO [Thread-4] org.apache.hadoop.hive.conf.HiveConf - Found configuration file file:/home/ort/spark/conf/hive-site.xml
2026-06-25T01:50:43,405 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.metastore.wm.default.pool.size does not exist
2026-06-25T01:50:43,406 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.llap.task.scheduler.preempt.independent does not exist
2026-06-25T01:50:43,406 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.llap.output.format.arrow does not exist
2026-06-25T01:50:43,406 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.tez.llap.min.reducer.per.executor does not exist
2026-06-25T01:50:43,406 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.arrow.root.allocator.limit does not exist
2026-06-25T01:50:43,406 WARN [Thread-4] org.apache.hadoop.hive.conf.HiveConf - HiveConf of name hive.vectorized.use.che

+------------------------------+--------------+------------------+---------------+----------+------------+-------------+
|festival_name                 |pais          |dist_aeropuerto_km|paises_directos|aerolineas|conectividad|accesibilidad|
+------------------------------+--------------+------------------+---------------+----------+------------+-------------+
|Hampton Court Palace Festival |United Kingdom|11.4              |77             |86        |2030.0      |1653.4       |
|Dance Valley                  |Netherlands   |17.0              |79             |80        |2002.0      |1493.8       |
|Crisis Festival               |Belgium       |5.8               |56             |63        |1392.0      |1247.9       |
|Festival Miami                |United States |8.3               |49             |43        |1378.0      |1182.8       |
|Rock Werchter                 |Belgium       |15.8              |56             |63        |1392.0      |1057.2       |
|Lokerse Feesten               |

In [4]:
q1.write.mode("overwrite").parquet(f"{ANL}/q1_accesibilidad")
pdf_q1 = spark.read.parquet(f"{ANL}/q1_accesibilidad").toPandas()

## Pregunta 2: países exportadores de artistas vs receptores de espectáculos 

In [12]:
spark.sql("USE festivales_aereo")

q2 = spark.sql("""
WITH intl AS (   -- solo flujos internacionales (origen != país del evento)
  SELECT artist_origin_country_id AS origin,
         event_country_id        AS dest,
         events_count
  FROM fact_country_music_flow
  WHERE artist_origin_country_id <> event_country_id
),
export AS (SELECT origin AS country_id, SUM(events_count) AS exportado FROM intl GROUP BY origin),
import AS (SELECT dest   AS country_id, SUM(events_count) AS recibido  FROM intl GROUP BY dest)
SELECT c.country_name                                   AS pais,
       COALESCE(e.exportado, 0)                         AS exportado,
       COALESCE(i.recibido, 0)                          AS recibido,
       COALESCE(e.exportado,0) - COALESCE(i.recibido,0) AS saldo_neto
FROM dim_country c
LEFT JOIN export e ON c.country_id = e.country_id
LEFT JOIN import i ON c.country_id = i.country_id
WHERE COALESCE(e.exportado,0) + COALESCE(i.recibido,0) > 0
""").cache()

print("== Top EXPORTADORES (más actuaciones de sus artistas en el exterior) ==")
q2.orderBy(F.desc("exportado")).show(15, truncate=False)

print("== Top RECEPTORES (más eventos de artistas extranjeros) ==")
q2.orderBy(F.desc("recibido")).show(15, truncate=False)

print("== Exportadores NETOS (saldo positivo) ==")
q2.orderBy(F.desc("saldo_neto")).show(10, truncate=False)

print("== Receptores NETOS (saldo negativo) ==")
q2.orderBy(F.asc("saldo_neto")).show(10, truncate=False)

== Top EXPORTADORES (más actuaciones de sus artistas en el exterior) ==


+--------------+---------+--------+----------+
|pais          |exportado|recibido|saldo_neto|
+--------------+---------+--------+----------+
|United States |14462    |10094   |4368      |
|United Kingdom|10912    |5837    |5075      |
|Canada        |3040     |3002    |38        |
|Sweden        |2562     |1115    |1447      |
|Germany       |2120     |8174    |-6054     |
|Australia     |1922     |1230    |692       |
|France        |1353     |2811    |-1458     |
|Japan         |1248     |865     |383       |
|Netherlands   |992      |3286    |-2294     |
|Ireland       |984      |441     |543       |
|Norway        |919      |397     |522       |
|Belgium       |839      |3273    |-2434     |
|Finland       |799      |756     |43        |
|Italy         |798      |661     |137       |
|Denmark       |547      |587     |-40       |
+--------------+---------+--------+----------+
only showing top 15 rows

== Top RECEPTORES (más eventos de artistas extranjeros) ==
+--------------+------

In [13]:
spark.sql("USE festivales_aereo")
q2_full = spark.sql("""
WITH intl AS (
  SELECT artist_origin_country_id AS origin, event_country_id AS dest, events_count
  FROM fact_country_music_flow WHERE artist_origin_country_id <> event_country_id),
export AS (SELECT origin AS country_id, SUM(events_count) AS exportado FROM intl GROUP BY origin),
import AS (SELECT dest   AS country_id, SUM(events_count) AS recibido  FROM intl GROUP BY dest)
SELECT c.country_name AS pais,
       COALESCE(e.exportado,0) AS exportado, COALESCE(i.recibido,0) AS recibido,
       COALESCE(e.exportado,0)-COALESCE(i.recibido,0) AS saldo_neto
FROM dim_country c
LEFT JOIN export e ON c.country_id=e.country_id
LEFT JOIN import i ON c.country_id=i.country_id
WHERE COALESCE(e.exportado,0)+COALESCE(i.recibido,0) > 0
""")
q2_full.write.mode("overwrite").parquet(f"{ANL}/q2_intercambio_paises")

spark.sql("DROP TABLE IF EXISTS anl_q2_intercambio_paises")
spark.sql(f"""CREATE EXTERNAL TABLE anl_q2_intercambio_paises
  (pais STRING, exportado BIGINT, recibido BIGINT, saldo_neto BIGINT)
  STORED AS PARQUET LOCATION '{ANL}/q2_intercambio_paises'""")
print("Q2 guardada en analytics + tabla Hive lista para Superset")



2026-06-25T01:53:06,773 INFO [Thread-4] org.apache.hadoop.hive.ql.security.authorization.plugin.sqlstd.SQLStdHiveAccessController - Created SQLStdHiveAccessController for session context : HiveAuthzSessionContext [sessionString=c9adafbc-c3f7-4579-ae8f-d280c1b39115, clientType=HIVECLI]
2026-06-25T01:53:06,775 WARN [Thread-4] org.apache.hadoop.hive.ql.session.SessionState - METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
2026-06-25T01:53:06,776 INFO [Thread-4] hive.metastore - Mestastore configuration hive.metastore.filter.hook changed from org.apache.hadoop.hive.metastore.DefaultMetaStoreFilterHookImpl to org.apache.hadoop.hive.ql.security.authorization.plugin.AuthorizationMetaStoreFilterHook
2026-06-25T01:53:06,780 INFO [Thread-4] hive.metastore - Closed a connection to metastore, current connections: 0
2026-06-25T01:53:06,781 INFO [Thread-4] hive.metastore - Trying to connect to metastore with URI thrift://l

## Pregunta 3: impacto de la temporada de festivales en aeropuertos cercanos de EE. UU.

In [5]:
spark.sql("USE festivales_aereo")

# 1) Aeropuertos cercanos a festivales de EE. UU. (aeropuerto más cercano de cada festival US)
spark.sql("""
  SELECT DISTINCT a.airport_id
  FROM fact_festival_air_accessibility a
  JOIN dim_festival f ON a.festival_id = f.festival_id
  JOIN dim_country  c ON f.country_id  = c.country_id
  WHERE a.airport_rank = 1 AND c.country_name = 'United States'
""").createOrReplaceTempView("festival_airports")
print("Aeropuertos cercanos a festivales US:", spark.table("festival_airports").count())

# 2) Comparación estacional: temporada alta (jun-ago) vs resto del año
q3 = spark.sql("""
WITH daily AS (
  SELECT d.airport_id,
         CAST(SUBSTR(CAST(d.date_id AS STRING), 5, 2) AS INT) AS mes,
         d.total_flights, d.avg_dep_delay, d.avg_arr_delay,
         d.cancellation_rate, d.delay_15_rate
  FROM fact_airport_daily_ops d
  JOIN festival_airports fa ON d.airport_id = fa.airport_id
)
SELECT
  CASE WHEN mes IN (6,7,8) THEN 'temporada alta (jun-ago)' ELSE 'resto del anio' END AS periodo,
  COUNT(*)                             AS dias_aeropuerto,
  ROUND(AVG(total_flights),1)          AS vuelos_diarios_prom,
  ROUND(AVG(avg_dep_delay),2)          AS demora_salida_min,
  ROUND(AVG(avg_arr_delay),2)          AS demora_llegada_min,
  ROUND(AVG(cancellation_rate)*100,2)  AS pct_cancelacion,
  ROUND(AVG(delay_15_rate)*100,2)      AS pct_demorados_15
FROM daily
GROUP BY CASE WHEN mes IN (6,7,8) THEN 'temporada alta (jun-ago)' ELSE 'resto del anio' END
ORDER BY periodo
""")
q3.show(truncate=False)

Aeropuertos cercanos a festivales US: 134


+------------------------+---------------+-------------------+-----------------+------------------+---------------+----------------+
|periodo                 |dias_aeropuerto|vuelos_diarios_prom|demora_salida_min|demora_llegada_min|pct_cancelacion|pct_demorados_15|
+------------------------+---------------+-------------------+-----------------+------------------+---------------+----------------+
|resto del anio          |35696          |110.0              |11.65            |6.12              |0.79           |17.73           |
|temporada alta (jun-ago)|12018          |113.4              |16.18            |13.76             |0.95           |23.57           |
+------------------------+---------------+-------------------+-----------------+------------------+---------------+----------------+



In [6]:
q3_mensual = spark.sql("""
SELECT CAST(SUBSTR(CAST(d.date_id AS STRING), 5, 2) AS INT) AS mes,
       ROUND(AVG(d.total_flights),1)         AS vuelos_diarios_prom,
       ROUND(AVG(d.avg_dep_delay),2)         AS demora_salida_min,
       ROUND(AVG(d.cancellation_rate)*100,2) AS pct_cancelacion,
       ROUND(AVG(d.delay_15_rate)*100,2)     AS pct_demorados_15
FROM fact_airport_daily_ops d
JOIN festival_airports fa ON d.airport_id = fa.airport_id
GROUP BY CAST(SUBSTR(CAST(d.date_id AS STRING), 5, 2) AS INT)
ORDER BY mes
""")
q3_mensual.show(12, truncate=False)


+---+-------------------+-----------------+---------------+----------------+
|mes|vuelos_diarios_prom|demora_salida_min|pct_cancelacion|pct_demorados_15|
+---+-------------------+-----------------+---------------+----------------+
|1  |104.6              |17.04            |1.72           |20.57           |
|2  |107.5              |12.24            |0.98           |16.97           |
|3  |112.7              |12.76            |0.82           |19.06           |
|4  |111.9              |10.67            |0.66           |17.64           |
|5  |112.9              |11.89            |0.48           |19.37           |
|6  |114.3              |16.74            |0.84           |24.27           |
|7  |114.3              |19.01            |1.21           |25.85           |
|8  |111.5              |12.81            |0.78           |20.62           |
|9  |108.5              |7.03             |0.38           |14.92           |
|10 |112.1              |6.63             |0.47           |14.74           |

In [7]:
q3_mensual.write.mode("overwrite").parquet(f"{ANL}/q3_estacionalidad_mensual")
pdf_q3 = spark.read.parquet(f"{ANL}/q3_estacionalidad_mensual").toPandas()

## Pregunta 4: ciudades referentes por género

In [14]:
spark.sql("USE festivales_aereo")

q4_pais = spark.sql("""
WITH gc AS (
  SELECT g.genre_name, co.country_name,
         COUNT(DISTINCT b.festival_id) AS festivales
  FROM bridge_festival_genre b
  JOIN dim_genre    g  ON b.genre_id = g.genre_id
  JOIN dim_festival f  ON b.festival_id = f.festival_id
  JOIN dim_country  co ON f.country_id = co.country_id
  GROUP BY g.genre_name, co.country_name
),
ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY genre_name ORDER BY festivales DESC, country_name) AS rn
  FROM gc
)
SELECT genre_name, country_name AS pais_referente, festivales
FROM ranked WHERE rn = 1
ORDER BY festivales DESC
LIMIT 20
""")
q4_pais.show(truncate=False)

+---------------------------+--------------+----------+
|genre_name                 |pais_referente|festivales|
+---------------------------+--------------+----------+
|música folk                |Germany       |8         |
|jazz                       |United States |7         |
|rock                       |United Kingdom|6         |
|heavy metal                |Germany       |4         |
|bluegrass                  |United States |3         |
|blues                      |Canada        |3         |
|música celta               |France        |3         |
|música clásica             |United States |3         |
|música de baile electrónica|Germany       |3         |
|música folclórica          |Canada        |3         |
|indie rock                 |United Kingdom|2         |
|música bretona             |France        |2         |
|música country             |Canada        |2         |
|Neue Deutsche Härte        |Germany       |2         |
|música del mundo           |France        |2   

In [15]:
q4_full = spark.sql("""
SELECT g.genre_name, co.country_name, COUNT(DISTINCT b.festival_id) AS festivales
FROM bridge_festival_genre b
JOIN dim_genre    g  ON b.genre_id = g.genre_id
JOIN dim_festival f  ON b.festival_id = f.festival_id
JOIN dim_country  co ON f.country_id = co.country_id
GROUP BY g.genre_name, co.country_name
""")
q4_full.write.mode("overwrite").parquet(f"{ANL}/q4_genero_pais")

spark.sql("DROP TABLE IF EXISTS anl_q4_genero_pais")
spark.sql(f"""CREATE EXTERNAL TABLE anl_q4_genero_pais
  (genre_name STRING, country_name STRING, festivales BIGINT)
  STORED AS PARQUET LOCATION '{ANL}/q4_genero_pais'""")
print("Q4 guardada en analytics + tabla Hive lista para Superset")


Q4 guardada en analytics + tabla Hive lista para Superset


## Pregunta 5: factores asociados a que Uruguay esté incluido en giras sudamericanas. 

In [8]:
spark.sql("USE festivales_aereo")
q5b = spark.sql("""
WITH sa AS (
  SELECT ae.artist_id, co.country_name AS country, ae.event_id
  FROM fact_artist_event ae
  JOIN dim_country co ON ae.event_country_id = co.country_id
  WHERE co.country_name IN ('Argentina','Bolivia','Brazil','Chile','Colombia','Ecuador',
                            'Guyana','Paraguay','Peru','Suriname','Uruguay','Venezuela')
    AND CAST(ae.event_date_id/10000 AS INT) >= 2021
),
por_artista AS (
  SELECT artist_id,
         MAX(CASE WHEN country='Uruguay'   THEN 1 ELSE 0 END) AS uy,
         COUNT(DISTINCT country)                              AS paises_sa,
         COUNT(DISTINCT event_id)                            AS eventos_sa,
         MAX(CASE WHEN country='Argentina' THEN 1 ELSE 0 END) AS arg,
         MAX(CASE WHEN country='Brazil'    THEN 1 ELSE 0 END) AS bra,
         MAX(CASE WHEN country='Chile'     THEN 1 ELSE 0 END) AS chi
  FROM sa GROUP BY artist_id
)
SELECT CASE WHEN uy=1 THEN 'incluyó Uruguay' ELSE 'no incluyó' END AS grupo,
       COUNT(*)                  AS artistas,
       ROUND(AVG(paises_sa),2)   AS paises_sa_prom,
       ROUND(AVG(eventos_sa),2)  AS eventos_sa_prom,
       ROUND(AVG(arg)*100,1)     AS pct_argentina,
       ROUND(AVG(bra)*100,1)     AS pct_brasil,
       ROUND(AVG(chi)*100,1)     AS pct_chile
FROM por_artista GROUP BY uy
""")
q5b.show(truncate=False)




+---------------+--------+--------------+---------------+-------------+----------+---------+
|grupo          |artistas|paises_sa_prom|eventos_sa_prom|pct_argentina|pct_brasil|pct_chile|
+---------------+--------+--------------+---------------+-------------+----------+---------+
|incluyó Uruguay|29      |1.41          |1.79           |41.4         |0.0       |0.0      |
|no incluyó     |411     |1.26          |2.1            |58.2         |38.2      |15.1     |
+---------------+--------+--------------+---------------+-------------+----------+---------+



In [9]:
print("Origen de los artistas que incluyeron Uruguay:")
spark.sql("""
SELECT co.country_name AS pais_origen, COUNT(DISTINCT t.artist_id) AS artistas
FROM fact_south_america_tours t
LEFT JOIN dim_country co ON t.artist_origin_country_id = co.country_id
WHERE t.visited_uruguay
GROUP BY co.country_name ORDER BY artistas DESC LIMIT 10
""").show(truncate=False)

Origen de los artistas que incluyeron Uruguay:
+-----------+--------+
|pais_origen|artistas|
+-----------+--------+
|Argentina  |22      |
|Uruguay    |7       |
+-----------+--------+



In [10]:
print("Por tipo de artista:")
spark.sql("""
SELECT a.artist_type,
       SUM(CASE WHEN t.visited_uruguay THEN 1 ELSE 0 END) AS con_uruguay,
       COUNT(*) AS total,
       ROUND(SUM(CASE WHEN t.visited_uruguay THEN 1 ELSE 0 END)/COUNT(*)*100,1) AS pct_incluye_uy
FROM fact_south_america_tours t
JOIN dim_artist a ON t.artist_id = a.artist_id
GROUP BY a.artist_type ORDER BY total DESC
""").show(truncate=False)


Por tipo de artista:
+-----------+-----------+-----+--------------+
|artist_type|con_uruguay|total|pct_incluye_uy|
+-----------+-----------+-----+--------------+
|Group      |21         |351  |6.0           |
|Person     |8          |170  |4.7           |
+-----------+-----------+-----+--------------+



In [11]:
q5b.write.mode("overwrite").parquet(f"{ANL}/q5_uruguay")
pdf_q5 = spark.read.parquet(f"{ANL}/q5_uruguay").toPandas()